# Multi-Tier Flood Simulation & Interactive Visualization — Trier, Germany (14-Day HQ100 Event)

This notebook demonstrates the end-to-end data processing, finite-state machine (FSM) simulation,
and interactive map visualization for the **Multi-Tier Robustness of Accessibility (RoA)** framework
over a 14-day, hourly (336-frame) HQ100 hydrodynamic flood event in Trier, Germany.

### Architecture & Processing Stages:
1. **Hydrodynamic Flood Stages:** 29 official Moselle gauge stages from 9.00 m to 11.80 m (HQ100 peak) with hourly interpolation across 336 frames.
2. **Infrastructure Interdependencies:** POI facilities (3 hospitals, 12 fire stations) linked to upstream power substations ($N=131$) and water facilities ($N=11$) via Network-Constrained Nearest-Neighbor (NCNN) routing.
3. **Dual-Resource Finite State Machine (FSM):** Independent power and water reserve buffers, depletion during outages, hysteresis-gated restarts, and cold-start reboot delays (`Operational` / `Depleting` / `Dead` / `Rebooting`).
4. **Dynamic RoA Computation:** Time-dependent graph routing from 500 origin sample points to active facilities, evaluating acute nadir loss and cumulative resilience ($RoA_{\text{Int}}$).
5. **Interactive 4-Panel Visualization Dashboard:** Standalone interactive HTML map (`trier_multi_tier_simulation_Trier_Germany.html`) embedding all 4 resilience tiers (**Level A**, **Level B1**, **Level B2**, **Level C**), dynamic $RoA(t)$ sparklines, and click-to-inspect facility diagnostics.

| Phase | Hours | Days | Flood level |
|---|---|---|---|
| Pre-event | 0–47 | 1–2 | 0 % (no flooding) |
| Rising water | 48–143 | 3–6 | 0 % → 100 % |
| HQ100 peak | 144–191 | 7–8 | 100 % (full flood) |
| Receding | 192–287 | 9–12 | 100 % → 0 % |
| Post-event | 288–335 | 13–14 | 0 % (no flooding) |

**Keyboard shortcuts** (once the interactive map is displayed):
* `Space` / `K` — play / pause
* `←` / `→` — step 1 h backward / forward
* `↑` / `↓` — step 1 d backward / forward


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import geopandas as gpd
import osmnx as ox
import pandas as pd
from shapely.ops import unary_union
from IPython.display import HTML, display

from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    RoaNotebookConfig,
    get_roa_cache_path,
    get_roa_hazard_data_path,
    get_roa_outputs_path,
    load_or_fetch_osm_features,
    set_notbook_wd,
)
from css_geodata_service.robustness_of_accessibility.utils.flood_interpolation import (
    SIMULATION_HOURS,
    _RISE_START, _RISE_END, _PEAK_END, _FALL_END,
    build_flood_animation_html,
    build_stage_for_hour,
    compute_hourly_flood_progress,
    load_or_compute_flood_stages,
    load_or_compute_hq_raw_flood_stages,
)
from css_geodata_service.robustness_of_accessibility.utils.flood_status import (
    compute_flood_status_by_stage,
    compute_dependency_status_by_stage,
    load_or_compute_flood_status_by_stage,
    load_or_compute_dependency_status_by_stage,
)
from css_geodata_service.robustness_of_accessibility.utils.ncnn import (
    load_or_calculate_ncnn_routes,
)
from css_geodata_service.robustness_of_accessibility.utils.backup_lifetime import (
    compute_backup_lifetime,
    load_or_compute_backup_lifetime,
)

from css_geodata_service.robustness_of_accessibility.robustness_of_accessibility import (
    load_or_draw_sample,
    prepare_services,
)
from css_geodata_service.robustness_of_accessibility.utils.stage_disruptions import (
    load_or_compute_stage_disruptions,
)
from css_geodata_service.robustness_of_accessibility.utils.dynamic_roa import (
    load_or_compute_dynamic_roa,
)
ox.settings.log_console = False
ox.settings.use_cache = True

print(f"osmnx     : {ox.__version__}")
print(f"geopandas : {gpd.__version__}")


osmnx     : 1.9.3
geopandas : 1.1.3


## 1. Configuration & Paths

In [2]:
set_notbook_wd()

place_name: str = RoaNotebookConfig.place_name   # "Trier, Germany"
event           = RoaNotebookConfig.event         # HQ100

cache_dir:       Path = get_roa_cache_path()
output_dir:      Path = get_roa_outputs_path()
hq_raw_dir:      Path = (Path.cwd().parent / "HQ_raw").resolve()

output_dir.mkdir(parents=True, exist_ok=True)

print(f"Place         : {place_name}")
print(f"Hazard event  : {event}")
print(f"HQ_raw dir    : {hq_raw_dir}")
print(f"Cache dir     : {cache_dir}")
print(f"Output dir    : {output_dir}")

Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Working dir set to: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code
Place         : Trier, Germany
Hazard event  : M
HQ_raw dir    : C:\Users\WelJo\IdeaProjects\forschungspraktikum\HQ_raw
Cache dir     : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\processed
Output dir    : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output


## 2. Study-Area Boundary

Loads the pre-processed administrative boundary of Trier from cache
(generated by `system_overview.ipynb`).  Falls back to OSM if the cache
file is absent.

In [3]:
boundary_cache = cache_dir / f"services/boundary_geom_{place_name}.geojson"

if boundary_cache.exists():
    print("Loading boundary from cache …")
    boundary_gdf  = gpd.read_file(boundary_cache)
    boundary_geom = unary_union(boundary_gdf.geometry)
else:
    print("Fetching boundary from OSM …")
    place_gdf     = ox.geocode_to_gdf(place_name)
    boundary_geom = unary_union(place_gdf.geometry)
    gpd.GeoDataFrame(geometry=[boundary_geom], crs="EPSG:4326").to_file(
        boundary_cache, driver="GeoJSON"
    )

minx, miny, maxx, maxy = boundary_geom.bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

print(f"Boundary loaded — centroid ({center_lat:.4f}°N, {center_lon:.4f}°E)")

Loading boundary from cache …
Boundary loaded — centroid (49.7779°N, 6.6495°E)


## 3. Hydrodynamic Flood Data (`HQ_raw`) & Stage Interpolation

Loads the official hydrodynamic Moselle gauge flood polygons from `HQ_raw/`
(Pegel 9.00 m up to 11.80 m for HQ100) via `load_or_compute_hq_raw_flood_stages`.

The 29 hydrodynamic gauge stages are pre-processed (CRS reprojection to EPSG:4326,
light coordinate simplification) and cached to disk as a single GeoJSON FeatureCollection
(`data/processed/flood_interpolation/`). Subsequent runs load the stages in milliseconds.

In [4]:
# --- Hydrodynamic Gauge Stages from HQ_raw (Official Moselle Gauge Data) ---
min_gauge_m = 9.00
max_gauge_m = 11.80  # Pegel 11.80m corresponds to HQ100 (100-year flood)

stages = load_or_compute_hq_raw_flood_stages(
    cache_dir=cache_dir,
    hq_raw_dir=hq_raw_dir,
    place_name=place_name,
    min_gauge_m=min_gauge_m,
    max_gauge_m=max_gauge_m,
)

non_empty = sum(1 for s in stages if s["geojson"] is not None)
print(f"HQ_raw flood stages ready    : {len(stages)} total ({min_gauge_m:.2f}m to {max_gauge_m:.2f}m), {non_empty} with polygon")


HQ_raw flood stages ready    : 29 total (9.00m to 11.80m), 29 with polygon


## 4. Facilities & Infrastructure — Direct Flood Status

Loads the same POI / infrastructure feature types already investigated in
`system_overview.ipynb` — hospitals, fire stations, power infrastructure
(substations + plants) and water infrastructure (water works + water towers
+ pumping stations) — from the shared cache (or fetches them from OSM on
first run).

For every pre-computed flood stage, each facility is checked
**independently** against that stage's flood polygon: a facility is
*directly flooded* when its own geometry intersects the flood polygon —
nothing more. Road accessibility is not used anywhere in this notebook.

This section only computes **direct** flood status. Section 5 below builds
on top of it — using the NCNN power/water connections — to determine when a
hospital or fire station also becomes unavailable *indirectly*, because its
connected infrastructure is flooded.

The flood-status logic itself lives in `utils/flood_status.py`
(`compute_flood_status_by_stage`) — this notebook only calls it and passes
the result on to the visualization. It does not implement the check itself.

In [5]:
services_cache_dir = cache_dir / "services"
services_cache_dir.mkdir(parents=True, exist_ok=True)

hospitals = load_or_fetch_osm_features(
    services_cache_dir / f"hospitals_{place_name}.geojson",
    boundary_geom, {"amenity": ["hospital"]}
)
fire_stations = load_or_fetch_osm_features(
    services_cache_dir / f"fire_stations_{place_name}.geojson",
    boundary_geom, {"amenity": ["fire_station"]}
)
power_substations = load_or_fetch_osm_features(
    services_cache_dir / f"power_substations_{place_name}.geojson",
    boundary_geom, {"power": ["substation"]}
)
power_plants = load_or_fetch_osm_features(
    services_cache_dir / f"power_plants_{place_name}.geojson",
    boundary_geom, {"power": ["plant"]}
)
water_towers = load_or_fetch_osm_features(
    services_cache_dir / f"water_towers_{place_name}.geojson",
    boundary_geom, {"man_made": ["water_tower"]}
)
water_works = load_or_fetch_osm_features(
    services_cache_dir / f"water_works_{place_name}.geojson",
    boundary_geom, {"man_made": ["water_works"]}
)
pumping_stations = load_or_fetch_osm_features(
    services_cache_dir / f"pumping_stations_{place_name}.geojson",
    boundary_geom, {"man_made": ["pumping_station"]}
)

# Power and water infrastructure are combined into single categories for this
# visualization — the same grouping already used for NCNN in system_overview.ipynb.
power_stations = gpd.GeoDataFrame(
    pd.concat([power_substations, power_plants], ignore_index=True), crs="EPSG:4326"
)
water_stations = gpd.GeoDataFrame(
    pd.concat([water_works, water_towers, pumping_stations], ignore_index=True), crs="EPSG:4326"
)

facility_gdfs = {
    "hospital":     hospitals,
    "fire_station": fire_stations,
    "power":        power_stations,
    "water":        water_stations,
}

print(f"Hospitals      : {len(hospitals)}")
print(f"Fire stations  : {len(fire_stations)}")
print(f"Power stations : {len(power_stations)}  (substations + plants)")
print(f"Water stations : {len(water_stations)}  (works + towers + pumping stations)")

Hospitals      : 3
Fire stations  : 12
Power stations : 131  (substations + plants)
Water stations : 11  (works + towers + pumping stations)


In [6]:
facility_flood_status = load_or_compute_flood_status_by_stage(
    cache_dir=cache_dir,
    facility_gdfs=facility_gdfs,
    stages=stages,
    place_name=place_name,
)

# Sanity check: nothing should be flooded at the pre-event stage (progress 0),
# and the peak-flood stage should have the highest (or equal) flooded count.
pre_event_stage_idx = min(range(len(stages)), key=lambda i: stages[i]["progress"])
peak_stage_idx = max(range(len(stages)), key=lambda i: stages[i]["progress"])

print(f"{'Facility type':14s}  {'Total':>5s}  {'Flooded @ pre-event':>20s}  {'Flooded @ peak':>15s}")
print("-" * 62)
for ftype, gdf in facility_gdfs.items():
    total = len(gdf)
    flooded_pre = sum(facility_flood_status[ftype][pre_event_stage_idx])
    flooded_peak = sum(facility_flood_status[ftype][peak_stage_idx])
    print(f"{ftype:14s}  {total:5d}  {flooded_pre:20d}  {flooded_peak:15d}")

Facility type   Total   Flooded @ pre-event   Flooded @ peak
--------------------------------------------------------------
hospital            3                     0                1
fire_station       12                     0                1
power             131                     0               25
water              11                     0                0


## 5. NCNN Infrastructure Connections & Cascading Failure

Hospitals and fire stations depend on power and water infrastructure to
function. This section determines when a POI becomes unavailable **not**
because it is itself flooded, but because the power or water station it
relies on is.

The POI → infrastructure assignment comes from the **Network-Constrained
Nearest-Neighbor (NCNN)** routes already computed in `system_overview.ipynb`
(`utils/ncnn.py`, `load_or_calculate_ncnn_routes`) — the nearest power/water
station reachable via the road network from each hospital/fire station.
Since this notebook builds `power_stations` / `water_stations` with the
exact same feature sets and ordering used there, calling
`load_or_calculate_ncnn_routes` again simply reuses the already-cached NCNN
result — it is not recomputed.

**Cascading rule** (implemented in `utils/flood_status.py`,
`compute_dependency_status_by_stage` — not in this notebook):

A hospital or fire station is **dead** at a given stage when *either*:
1. its own geometry is directly covered by the flood polygon (section 4), **or**
2. the power station connected to it (via NCNN) is dead at that stage, **or**
3. the water station connected to it (via NCNN) is dead at that stage.

A power/water station's own dead status is still purely direct-flood-based
(section 4) — this section does not introduce any further cascading beyond
POI ← infrastructure (e.g. no dependency *between* power and water
stations, and no road-accessibility reasoning).

In [7]:
network_cache = cache_dir / f"network/drive_graph_{place_name}.graphml"

if network_cache.exists():
    print("Loading road network from cache …")
    road_network = ox.load_graphml(network_cache)
else:
    print("Downloading road network from OSM (may take ~1–2 min) …")
    road_network = ox.graph_from_polygon(polygon=boundary_geom, network_type=RoaNotebookConfig.network_type)
    network_cache.parent.mkdir(parents=True, exist_ok=True)
    ox.save_graphml(road_network, filepath=network_cache)

# Undirected so one-way restrictions don't block underground infrastructure
# paths (power/water pipes are bidirectional by nature) — same choice made
# for the NCNN computation in system_overview.ipynb.
road_network_undirected = road_network.to_undirected()

print(f"Road network nodes : {road_network.number_of_nodes():,}")
print(f"Road network edges : {road_network.number_of_edges():,}")

Loading road network from cache …
Road network nodes : 5,817
Road network edges : 12,687


In [8]:
poi_gdfs = {
    "hospital":     hospitals,
    "fire_station": fire_stations,
}
infrastructure_gdfs = {
    "power": power_stations,
    "water": water_stations,
}

print("Computing / loading NCNN routes (POI → nearest infrastructure via road network) …")
ncnn_results = load_or_calculate_ncnn_routes(
    cache_dir=cache_dir,
    poi_gdfs=poi_gdfs,
    infrastructure_gdfs=infrastructure_gdfs,
    street_network=road_network_undirected,
    place_name=place_name,
)

print(f"{'POI type':15s}  {'Infra type':10s}  {'POIs':>4s}  {'Mean (m)':>9s}  {'Max (m)':>8s}")
print("-" * 57)
for poi_type, infra_dict in ncnn_results.items():
    for infra_type, gdf in infra_dict.items():
        finite = gdf[gdf["route_length_m"] < float("inf")]["route_length_m"]
        mean_m = f"{finite.mean():.0f}" if len(finite) else "—"
        max_m  = f"{finite.max():.0f}"  if len(finite) else "—"
        print(f"{poi_type:15s}  {infra_type:10s}  {len(gdf):>4d}  {mean_m:>9s}  {max_m:>8s}")

Computing / loading NCNN routes (POI → nearest infrastructure via road network) …
POI type         Infra type  POIs   Mean (m)   Max (m)
---------------------------------------------------------
hospital         power          3        250       424
hospital         water          3       2186      2614
fire_station     power         12        749      3789
fire_station     water         12       2444      5800


In [9]:
dependency_status = load_or_compute_dependency_status_by_stage(
    cache_dir=cache_dir,
    infrastructure_gdfs=infrastructure_gdfs,
    connections=ncnn_results,
    direct_flooded_by_stage=facility_flood_status,
    place_name=place_name,
)

# Sanity check: for hospitals/fire stations, the combined (direct + cascading)
# dead count at peak flood should be >= the direct-only count from section 4 —
# cascading failure can only add dead facilities, never remove them.
print(f"{'POI type':14s}  {'Direct @ peak':>13s}  {'Direct+Cascading @ peak':>24s}")
print("-" * 55)
for poi_type in ncnn_results:
    direct_peak = sum(facility_flood_status[poi_type][peak_stage_idx])
    combined_peak = sum(dependency_status[poi_type]["dead_by_stage"][peak_stage_idx])
    print(f"{poi_type:14s}  {direct_peak:13d}  {combined_peak:24d}")
    assert combined_peak >= direct_peak, "cascading failure must not reduce the dead count"

POI type        Direct @ peak   Direct+Cascading @ peak
-------------------------------------------------------
hospital                    1                         1
fire_station                1                         2


## 6. Backup Lifetime — Dual-Resource Finite State Machine

Section 5 gave every hospital/fire station a binary dead/alive status per flood *stage*
(direct flooding OR its nearest power/water station being flooded). This section replaces that
binary status — for hospitals and fire stations only; power/water stations keep the simple
direct-flood status from Section 4, per Logic.md §6 — with the full mechanism from
[`plan/Logic.md`](plan/Logic.md):

* Each facility tracks **independent power and water reserve buffers**, decaying at
  `loss_rate` while disconnected and refilling at `gain_rate` while connected.
* A facility stays `Operational`/`Depleting` as long as **Standard Viability** holds
  (each resource is either connected, or its buffer hasn't hit 0%).
* It goes `Dead` the instant that fails — and, unlike Section 5's model, **direct flooding
  overrides everything**: a submerged building is `Dead` regardless of any reserve, exactly as
  before, just now expressed as one guard in a richer state machine instead of the only rule.
* From `Dead`, it only starts `Rebooting` once **Restart Viability** holds — connected, or a
  disconnected buffer above the `restart_threshold` hysteresis guard (Logic.md §2, §5) — and
  only returns to service after `recharge_delay` ticks in `Rebooting`.

**Resolution note:** buffer depletion is an inherently hourly process, but Sections 4–5 only
computed status at **stage** resolution (50 discrete flood levels spread across 336 hours).
`build_stage_for_hour` — the same lookup `build_flood_animation_html` already uses internally
to pick which flood polygon to draw each frame — expands that stage-resolution connectivity
data to hourly before it's fed into the state machine, so both stay consistent.

In [10]:
# Per-facility-type backup parameters (Logic.md §4). Kept here, not in
# utils/backup_lifetime.py, so they're easy to tune without touching the
# simulation engine itself.
BACKUP_CFG = {
    "hospital": {
        "recharge_delay": 48,  # hours — cold-start delay once Restart Viability holds
        "resources": {
            "power": {"capacity": 72, "loss_rate": 1, "gain_rate": 1},
            "water": {"capacity": 48, "loss_rate": 1, "gain_rate": 1},
        },
    },
    "fire_station": {
        "recharge_delay": 24,
        "resources": {
            "power": {"capacity": 24, "loss_rate": 1, "gain_rate": 1},
            "water": {"capacity": 12, "loss_rate": 1, "gain_rate": 1},
        },
    },
}
RESTART_THRESHOLD = 0.15  # Logic.md §4/§5 hysteresis guard (15%)

# Shape/color vocabulary for the map rendering (visual.md §4-§6) — passed
# straight through to build_flood_animation_html in Section 8.
BACKUP_SHAPE = {"hospital": "circle", "fire_station": "triangle"}
RESOURCE_RING_META = {
    "power": {"label": "Power", "color": "#FF8C00"},
    "water": {"label": "Water", "color": "#00AEEF"},
}

stage_for_hour = build_stage_for_hour(stages)
print(f"Hours simulated : {len(stage_for_hour)}")
print(f"Stages spanned  : {len(set(stage_for_hour))} of {len(stages)}")

print("\nSimulating backup lifetime (Operational / Depleting / Dead / Rebooting) …")
backup_lifetime = load_or_compute_backup_lifetime(
    cache_dir=cache_dir,
    poi_types=["hospital", "fire_station"],
    direct_flooded_by_stage=facility_flood_status,
    dependency_status=dependency_status,
    stage_for_hour=stage_for_hour,
    backup_cfg=BACKUP_CFG,
    restart_threshold=RESTART_THRESHOLD,
    place_name=place_name,
)
print("Done.")


Hours simulated : 336
Stages spanned  : 29 of 29

Simulating backup lifetime (Operational / Depleting / Dead / Rebooting) …
Done.


**Sanity checks.** Two things must always hold by construction — worth verifying against
the real data rather than trusting the implementation blindly:

1. A directly flooded facility is `Dead` at every hour it's flooded, full stop — no reserve
   buffer keeps a submerged building running (Logic.md §3).
2. Since the flood event lasts 14 days while the configured reserves/reboot delays are on the
   order of 1–3 days, at least some facilities should be seen `Rebooting` at some point.

What should **not** be expected: a simple ordering between this section's `Dead` counts and
Section 5's direct+cascading counts at any given hour. Backup reserves *delay* death relative
to Section 5's instantaneous model (so the new model can show *fewer* dead at a given hour),
while the hysteresis/reboot-delay guard *delays recovery* relative to it (so the new model can
also show *more* dead once a stage's flood recedes but a facility hasn't finished rebooting
yet). Both directions are correct, intended behaviour — that's precisely what "Dual-Resource
Tracking" over a memoryless per-stage check is supposed to add.

In [11]:
_CODE = {"Operational": "O", "Depleting": "D", "Rebooting": "R", "Dead": "X"}
peak_hour = max(range(SIMULATION_HOURS), key=compute_hourly_flood_progress)

print(f"{'POI type':14s}  {'Facilities':>10s}  {'Dead @ peak-h':>13s}  {'Direct+Cascading @ peak-h':>26s}  {'Ever Rebooting':>15s}")
print("-" * 90)
for poi_type, sims in backup_lifetime.items():
    n = len(sims)
    dead_at_peak = sum(1 for sim in sims if sim["state_by_hour"][peak_hour] == "Dead")
    ever_rebooting = sum(1 for sim in sims if "Rebooting" in sim["state_by_hour"])
    direct_dead_at_peak = sum(1 for row in dependency_status[poi_type]["dead_by_stage"][stage_for_hour[peak_hour]] if row)
    print(f"{poi_type:14s}  {n:10d}  {dead_at_peak:13d}  {direct_dead_at_peak:26d}  {ever_rebooting:15d}")

    for i, sim in enumerate(sims):
        for h in range(SIMULATION_HOURS):
            if facility_flood_status[poi_type][stage_for_hour[h]][i]:
                assert sim["state_by_hour"][h] == "Dead", (
                    f"{poi_type} #{i} is directly flooded at hour {h} but not marked Dead"
                )

print()
print(f"{'Name':24s}  {'Type':13s}  {'Oper.':>6s}  {'Deplet.':>7s}  {'Reboot.':>7s}  {'Dead':>6s}")
print("-" * 76)
_STATES = ["Operational", "Depleting", "Rebooting", "Dead"]
for poi_type, sims in backup_lifetime.items():
    gdf = facility_gdfs[poi_type]
    for i, sim in enumerate(sims):
        name = gdf.iloc[i].get("name") if "name" in gdf.columns else None
        name = str(name) if name is not None and pd.notna(name) else f"{poi_type} #{i}"
        counts = {s: sim["state_by_hour"].count(s) for s in _STATES}
        print(
            f"{name[:24]:24s}  {poi_type:13s}  "
            + "  ".join(f"{counts[s]:7d}" for s in _STATES)
        )


POI type        Facilities  Dead @ peak-h   Direct+Cascading @ peak-h   Ever Rebooting
------------------------------------------------------------------------------------------
hospital                 3              1                           1                1
fire_station            12              2                           2                2

Name                      Type            Oper.  Deplet.  Reboot.    Dead
----------------------------------------------------------------------------
Klinikum Mutterhaus der   hospital           336        0        0        0
Klinikum Mutterhaus der   hospital           336        0        0        0
Krankenhaus der Barmherz  hospital           230        0       47       59
Freiwillige Feuerwehr Tr  fire_station       336        0        0        0
Freiwillige Feuerwehr Tr  fire_station       336        0        0        0
Freiwillige Feuerwehr Tr  fire_station       108       26       27      175
Freiwillige Feuerwehr Tr  fire_station  

## 7. Dynamic Robustness of Accessibility (RoA) Computation

Extends the static physical accessibility metric from Kaub et al. (2024) to a
time-dependent disaster setting:

$$\text{RoA}(t) = \sum_{j=1}^{m} a_j \cdot \frac{c(s_j, d_j, t_0)}{c'(s_j, d_j'(t), t)}$$

* **Fixed Spatial Origin Sampling ($s \in \mathcal{S}$):** Loads or draws 500 origin sample points
  across Trier's residential boundary (persisted to `data/processed/samples/` via `load_or_draw_sample`).
* **Stage-Based Road Network Pre-Slicing:** Intersects the road network with the flood stages
  upfront and caches flooded edges to `data/processed/network/` via `load_or_compute_stage_disruptions`.
* **Dynamic Routing to Active Facilities $\mathcal{D}_{\text{active}}(t)$:**
  At each hour $t \in [0, 335]$, Dijkstra shortest paths are evaluated to whichever facilities are
  functional (`Operational` or `Depleting`). If a facility dies or floods, routes automatically divert
  to the next surviving facility, capturing detours and service degradation at full hourly resolution.
* **Integrated Crisis Resilience ($\text{RoA}_{\text{Int}}$):** Integrates $\text{RoA}(t)$ across the
  entire 14-day horizon:
  $$\text{RoA}_{\text{Int}} = \frac{1}{336} \int_0^{336} \text{RoA}(t) \, dt$$


In [12]:
# 1. Fixed spatial origin sampling (m = 500 citizen points across Trier)
samples = load_or_draw_sample(
    cache_path=cache_dir / f"samples/samples_{place_name}_500.geojson",
    polygon=boundary_geom,
    gdf_nodes_drive_service_graph=ox.graph_to_gdfs(road_network_undirected, edges=False),
    number_total_samples=500,
)

# 2. Stage-based road network disruption caching (discrete flood stages)
stage_disruptions = load_or_compute_stage_disruptions(
    cache_dir=cache_dir,
    street_network=road_network_undirected,
    stages=stages,
    place_name=place_name,
)

# 3. Prepare destination POIs (bind nearest street network nodes)
hospitals_prep = prepare_services(hospitals, road_network_undirected)
fire_stations_prep = prepare_services(fire_stations, road_network_undirected)
poi_gdfs_prepared = {
    "hospital": hospitals_prep,
    "fire_station": fire_stations_prep,
}

# 4. Compute dynamic hourly RoA and time-integrated resilience RoA_Int
roa_results = load_or_compute_dynamic_roa(
    cache_dir=cache_dir,
    street_network=road_network_undirected,
    samples=samples,
    poi_gdfs=poi_gdfs_prepared,
    backup_lifetime_results=backup_lifetime,
    stage_disruptions=stage_disruptions,
    stages=stages,
    place_name=place_name,
)

# 5. Display evaluation metrics
print("=" * 65)
print("  DYNAMIC ROBUSTNESS OF ACCESSIBILITY (RoA) EVALUATION")
print("=" * 65)
print(f"  Origin sample points  : {roa_results['n_samples']}")
print(f"  Simulation horizon    : {len(roa_results['hours'])} hours (14 days)")
print()
print(f"  {'Service Type':18s}  {'Baseline (t0)':>15s}  {'Peak Loss':>12s}  {'Resilience (RoA_Int)':>20s}")
print("-" * 65)
for ptype in ['hospital', 'fire_station']:
    scores = roa_results['roa_by_type'][ptype]
    int_score = roa_results['roa_int_by_type'][ptype] * 100
    peak_score = min(scores) * 100
    peak_loss = 100.0 - peak_score
    print(f"  {ptype.replace('_', ' ').title():16s}  {scores[0]*100:13.1f}%  {-peak_loss:11.1f}%  {int_score:19.2f}%")
print("-" * 65)
comb_int = roa_results['roa_int_combined'] * 100
comb_peak_loss = 100.0 - min(roa_results['roa_combined']) * 100
print(f"  {'Combined Total':16s}  {roa_results['roa_combined'][0]*100:13.1f}%  {-comb_peak_loss:11.1f}%  {comb_int:19.2f}%")
print("=" * 65)


C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\robustness_of_accessibility.py:97: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  services["position"] = services.geometry.centroid
C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\robustness_of_accessibility.py:97: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  services["position"] = services.geometry.centroid


  DYNAMIC ROBUSTNESS OF ACCESSIBILITY (RoA) EVALUATION
  Origin sample points  : 164
  Simulation horizon    : 336 hours (14 days)

  Service Type          Baseline (t0)     Peak Loss  Resilience (RoA_Int)
-----------------------------------------------------------------
  Hospital                   94.7%        -59.1%                79.13%
  Fire Station               96.0%        -30.1%                87.40%
-----------------------------------------------------------------
  Combined Total             95.4%        -44.6%                83.27%


## 8. Simulation Schedule Overview


In [13]:
print("=" * 56)
print("  FLOOD SIMULATION SCHEDULE — 14-day HQ100 event")
print("=" * 56)
print(f"  Total frames  : {SIMULATION_HOURS} (one per hour)")
print(f"  Flood stages  : {len(stages)} pre-computed levels")
print()
print(f"  Phase 1 — Pre-event    : Hours   0–{_RISE_START - 1:3d}  (Days  1–2 )")
print(f"  Phase 2 — Rising water : Hours {_RISE_START:3d}–{_RISE_END  - 1:3d}  (Days  3–6 )")
print(f"  Phase 3 — HQ100 peak   : Hours {_RISE_END:3d}–{_PEAK_END  - 1:3d}  (Days  7–8 )")
print(f"  Phase 4 — Receding     : Hours {_PEAK_END:3d}–{_FALL_END  - 1:3d}  (Days  9–12)")
print(f"  Phase 5 — Post-event   : Hours {_FALL_END:3d}–{SIMULATION_HOURS - 1:3d}  (Days 13–14)")
print("=" * 56)
print()

# Symmetry check
p_h0   = compute_hourly_flood_progress(0)
p_h96  = compute_hourly_flood_progress(96)   # midway through rising phase
p_h168 = compute_hourly_flood_progress(168)  # midway through peak
p_h240 = compute_hourly_flood_progress(240)  # midway through recession
p_h335 = compute_hourly_flood_progress(335)
print("  Symmetry check (rising vs. receding should mirror each other):")
print(f"    Hour   0 (pre)         : {p_h0:.3f}")
print(f"    Hour  96 (mid-rise)    : {p_h96:.3f}")
print(f"    Hour 168 (mid-peak)    : {p_h168:.3f}")
print(f"    Hour 240 (mid-recede)  : {p_h240:.3f}  ← should equal mid-rise")
print(f"    Hour 335 (post)        : {p_h335:.3f}")

  FLOOD SIMULATION SCHEDULE — 14-day HQ100 event
  Total frames  : 336 (one per hour)
  Flood stages  : 29 pre-computed levels

  Phase 1 — Pre-event    : Hours   0– 47  (Days  1–2 )
  Phase 2 — Rising water : Hours  48–143  (Days  3–6 )
  Phase 3 — HQ100 peak   : Hours 144–191  (Days  7–8 )
  Phase 4 — Receding     : Hours 192–287  (Days  9–12)
  Phase 5 — Post-event   : Hours 288–335  (Days 13–14)

  Symmetry check (rising vs. receding should mirror each other):
    Hour   0 (pre)         : 0.000
    Hour  96 (mid-rise)    : 0.500
    Hour 168 (mid-peak)    : 1.000
    Hour 240 (mid-recede)  : 0.500  ← should equal mid-rise
    Hour 335 (post)        : 0.000


## 9. Build & Launch Multi-Tier Flood Simulation Dashboard

Generates the **self-contained Multi-Tier HTML animation dashboard** and displays it directly in the notebook.
All pre-computed flood polygons, facility markers, NCNN power/water routes, the full 4-tier simulation bundle
(Levels A, B1, B2, and C), and dual-resource state machines are embedded into a single standalone HTML file.

The HTML file is saved to:
* `data/output/trier_multi_tier_simulation_Trier_Germany.html`

and can be opened directly in any web browser for full-screen interactive presentations.

**Interactive 4-Panel UI:**

* **Top-Left (Scenario Control):** Live radio buttons to hot-swap between **Level A (Road Baseline)**, **Level B1 (Power Cascade)**, **Level B2 (Water Cascade)**, and **Level C (Compound Failure)** without page reload.
* **Bottom-Left (Resilience Analytics):** Dynamic $RoA(t)$ percentage, total $RoA_{\text{Int}}$ badge, hospital vs. fire station breakdown, and animated SVG sparkline timeline with synchronized playhead.
* **Top-Right (Infrastructure Status):** Active layer swatches (Substations, Water Works, Power & Water Routes) and aggregate facility state counters (Operational, Depleting, Dead, Rebooting).
* **Bottom-Right (Facility Inspector):** Click any hospital or fire station on the map to inspect real-time Power/Water buffer levels, decay/recharge status, and restart viability guards.

**In-player controls:** play/pause (`Space`/`K`), ±1h (`←`/`→`), ±1d (`↑`/`↓`), time slider scrubber, and variable playback speed selector.

In [14]:
import json

# Power/water stations keep the plain marker treatment (Section 4 direct flood status)
FACILITY_STYLE = {
    "power": {"label": "Substations & Plants", "color": "#e67e22"},
    "water": {"label": "Water Works & Towers", "color": "#3498db"},
}
facility_layers = {
    ftype: {
        "gdf": facility_gdfs[ftype],
        "flooded_by_stage": facility_flood_status[ftype],
        "label": style["label"],
        "color": style["color"],
    }
    for ftype, style in FACILITY_STYLE.items()
}

# Load pre-cached multi-tier simulation bundle (Levels A, B1, B2, C)
safe_place = place_name.replace(" ", "_").replace(",", "")
bundle_path = cache_dir / "roa" / f"multi_tier_roa_bundle_{safe_place}_336h.json"
if bundle_path.exists():
    with open(bundle_path, "r", encoding="utf-8") as f:
        multi_tier_bundle = json.load(f)
else:
    multi_tier_bundle = None

# Initial backup layers and RoA data (using Level_C compound failure from bundle)
level_c_backup = (
    multi_tier_bundle["tiers"]["Level_C"]["backup_lifetime"]
    if multi_tier_bundle
    else backup_lifetime
)
initial_roa = (
    multi_tier_bundle["tiers"]["Level_C"]["roa"]
    if multi_tier_bundle
    else roa_results
)

# Hospitals/fire stations move to the backup-lifetime rendering
# (Operational/Depleting/Rebooting/Dead state machine).
backup_layers = {
    poi_type: {
        "gdf": facility_gdfs[poi_type],
        "shape": BACKUP_SHAPE[poi_type],
        "cfg": BACKUP_CFG[poi_type],
        "backup": level_c_backup[poi_type],
    }
    for poi_type in level_c_backup
}

# Connection lines driven by the per-stage dependency status
CONNECTION_STYLE = {
    "power": {"label": "Power Routes", "color": "#e67e22"},
    "water": {"label": "Water Routes", "color": "#3498db"},
}
connection_layers = {
    infra_type: {
        "label": style["label"],
        "color": style["color"],
        "poi_connections": {
            poi_type: {
                "gdf": ncnn_results[poi_type][infra_type],
                "dead_by_stage": dependency_status[poi_type]["connections"][infra_type]["dead_by_stage"],
            }
            for poi_type in dependency_status
        },
    }
    for infra_type, style in CONNECTION_STYLE.items()
}

output_html_path = output_dir / f"trier_multi_tier_simulation_{safe_place}.html"
html_path = build_flood_animation_html(
    boundary_geom=boundary_geom,
    stages=stages,
    output_path=output_html_path,
    center_lat=center_lat,
    center_lon=center_lon,
    facility_layers=facility_layers,
    connection_layers=connection_layers,
    backup_layers=backup_layers,
    resource_ring_meta=RESOURCE_RING_META,
    restart_threshold=RESTART_THRESHOLD,
    roa_data=initial_roa,
    multi_tier_bundle=multi_tier_bundle,
)

print(f"Multi-Tier Animation Dashboard : {html_path}")
print(f"File size                      : {html_path.stat().st_size / 1024:.0f} KB")
print()
print("Open the HTML file directly in a browser for the interactive 4-panel multi-tier presentation.")


Multi-Tier Animation Dashboard : C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output\trier_multi_tier_simulation_Trier_Germany.html
File size                      : 5625 KB

Open the HTML file directly in a browser for the interactive 4-panel multi-tier presentation.


In [15]:
import base64

html_bytes = html_path.read_bytes()
html_b64 = base64.b64encode(html_bytes).decode("ascii")

display(HTML(
    f'<iframe src="data:text/html;base64,{html_b64}" width="100%" height="780px" '
    f'frameborder="0" '
    f'style="border-radius:8px; box-shadow:0 2px 14px rgba(0,0,0,0.2);">'
    f'</iframe>'
))

print(f"\nFor full-screen interactive use, open directly in a browser:\n  {html_path}")


C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\.venv\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")



For full-screen interactive use, open directly in a browser:
  C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\data\output\trier_multi_tier_simulation_Trier_Germany.html
